# Evaluation - Invitation Triage

This notebook evaluates the performance of the triage system against marking a calendar invite.

In [1]:
from typing import Any

import pandas as pd
from fuzzywuzzy import fuzz

# Import the test dataset
from triage_dataset import TEST_PERSONA, TRIAGE_TEST_CASES

from box2.triage.models import MinisterPersona, TriagedDecision
from box2.triage.triage import triage_invitation

## Helper Functions

In [2]:
def fuzzy_match(expected: str | None, actual: str | None, threshold: int = 80) -> bool:
    """Check if two strings are 'close enough' using fuzzy matching."""
    if not expected and not actual:
        return True
    if not expected or not actual:
        return False

    return fuzz.partial_ratio(str(expected).lower(), str(actual).lower()) >= threshold


def check_reasoning_mentions(reasoning: str, expected_points: list[str], threshold: int = 70) -> dict[str, Any]:
    """
    Check if the reasoning mentions expected key points using fuzzy matching.

    Args:
        reasoning: The actual reasoning text from the agent
        expected_points: List of key phrases that should appear in reasoning
        threshold: Fuzzy match threshold (default 70 for more lenient matching)

    Returns:
        Dict with coverage metrics and missing points
    """
    if not expected_points:
        return {
            "coverage": 1.0,
            "mentions_found": 0,
            "mentions_expected": 0,
            "missing_points": [],
        }

    reasoning_lower = reasoning.lower()
    mentions_found = []
    missing_points = []

    for point in expected_points:
        point_lower = point.lower()

        # Check for fuzzy match anywhere in the reasoning
        # Split reasoning into sentences/phrases for better matching
        found = False
        for sentence in reasoning_lower.split("."):
            if fuzz.partial_ratio(point_lower, sentence.strip()) >= threshold:
                found = True
                break

        if found:
            mentions_found.append(point)
        else:
            missing_points.append(point)

    coverage = len(mentions_found) / len(expected_points) if expected_points else 0.0

    return {
        "coverage": coverage,
        "mentions_found": len(mentions_found),
        "mentions_expected": len(expected_points),
        "missing_points": missing_points,
    }


def evaluate_single_triage(test_case: dict, result: TriagedDecision) -> dict[str, Any]:
    """
    Evaluate a single triage result against ground truth.

    Args:
        test_case: Test case dictionary with invitation and expected results
        result: TriagedDecision from the triage agent

    Returns:
        Dictionary with evaluation metrics
    """
    test_id = test_case["test_id"]
    expected = test_case["expected"]

    evaluation = {
        "test_id": test_id,
        "description": test_case["description"],
        "expected_decision": expected["decision"],
        "actual_decision": result.decision,
        "expected_priority": expected["priority"],
        "actual_priority": result.priority,
    }

    # 1. DECISION CORRECTNESS
    evaluation["decision_correct"] = expected["decision"].lower() == result.decision.lower()

    # 2. PRIORITY CORRECTNESS
    evaluation["priority_correct"] = expected["priority"].lower() == result.priority.lower()

    # 3. CALENDAR MENTION
    expected_calendar_mention = expected.get("should_mention_calendar", False)
    # Check if reasoning mentions calendar/schedule/conflict/available
    calendar_keywords = [
        "calendar",
        "schedule",
        "conflict",
        "available",
        "clash",
        "timing",
        "slot",
        "commitment",
        "meeting",
    ]
    actual_mentions_calendar = any(keyword in result.reason.lower() for keyword in calendar_keywords)

    evaluation["expected_calendar_mention"] = expected_calendar_mention
    evaluation["actual_calendar_mention"] = actual_mentions_calendar
    evaluation["calendar_mention_correct"] = expected_calendar_mention == actual_mentions_calendar

    # 4. REASONING QUALITY
    expected_points = expected.get("key_reasoning_points", [])
    reasoning_analysis = check_reasoning_mentions(result.reason, expected_points)

    evaluation["reasoning_coverage"] = reasoning_analysis["coverage"]
    evaluation["reasoning_points_found"] = reasoning_analysis["mentions_found"]
    evaluation["reasoning_points_expected"] = reasoning_analysis["mentions_expected"]
    evaluation["reasoning_missing_points"] = reasoning_analysis["missing_points"]

    # Good reasoning = covers at least 50% of expected points
    evaluation["reasoning_quality_good"] = reasoning_analysis["coverage"] >= 0.5

    # 5. DRAFT RESPONSE PRESENT
    evaluation["has_draft_response"] = bool(result.draft_response)
    evaluation["draft_response_length"] = len(result.draft_response) if result.draft_response else 0

    # 6. OVERALL CORRECTNESS
    # Decision is most important, then priority, then reasoning quality
    evaluation["core_correct"] = evaluation["decision_correct"] and evaluation["priority_correct"]

    evaluation["fully_correct"] = (
        evaluation["decision_correct"] and evaluation["priority_correct"] and evaluation["reasoning_quality_good"]
    )

    return evaluation

# Calculation Metrics

In [3]:
async def run_triage_evaluation(test_cases: list[dict], persona: MinisterPersona) -> pd.DataFrame:
    """
    Run evaluation on all triage test cases.

    Args:
        test_cases: List of test case dictionaries
        persona: Minister persona to use for triage

    Returns:
        DataFrame with evaluation results
    """
    results = []

    print(f"Running triage evaluation on {len(test_cases)} test cases...")
    print("=" * 80)

    for i, test_case in enumerate(test_cases, 1):
        test_id = test_case["test_id"]
        description = test_case["description"]

        print(f"\n[{i}/{len(test_cases)}] {test_id}: {description}")

        try:
            # Run triage
            invitation = test_case["invitation"]
            result = await triage_invitation(invitation, persona)

            # Evaluate result
            evaluation = evaluate_single_triage(test_case, result)

            # Add actual outputs for debugging
            evaluation["actual_reason"] = result.reason
            evaluation["actual_draft_response"] = result.draft_response

            results.append(evaluation)

            # Print result
            if evaluation["fully_correct"]:
                print(f"  ✓ PERFECT: Decision={result.decision}, Priority={result.priority}")
            elif evaluation["core_correct"]:
                print(f"  ✓ CORE CORRECT: Decision={result.decision}, Priority={result.priority}")
                print(f"  ⚠ Reasoning coverage: {evaluation['reasoning_coverage']:.0%}")
            else:
                print("  ✗ INCORRECT:")
                if not evaluation["decision_correct"]:
                    print(f"    Decision: Expected '{test_case['expected']['decision']}', got '{result.decision}'")
                if not evaluation["priority_correct"]:
                    print(f"    Priority: Expected '{test_case['expected']['priority']}', got '{result.priority}'")

        except Exception as e:
            print(f"  ✗ ERROR: {str(e)}")
            results.append(
                {
                    "test_id": test_id,
                    "description": test_case["description"],
                    "error": str(e),
                    "decision_correct": False,
                    "priority_correct": False,
                    "core_correct": False,
                    "fully_correct": False,
                }
            )

    print("\n" + "=" * 80)
    print("Evaluation complete!")

    return pd.DataFrame(results)


def calculate_triage_metrics(eval_df: pd.DataFrame) -> dict[str, Any]:
    """Calculate overall metrics for triage performance."""

    total = len(eval_df)
    if total == 0:
        return {"note": "No test cases evaluated", "n": 0}

    # Remove error cases for metric calculation
    valid_df = eval_df[~eval_df.get("error", pd.Series([None] * len(eval_df))).notna()].copy()
    n_valid = len(valid_df)

    if n_valid == 0:
        return {"note": "All test cases had errors", "n": 0, "n_errors": total}

    metrics = {
        "n": n_valid,
        "n_errors": total - n_valid,
        # Primary metrics
        "decision_accuracy": valid_df["decision_correct"].mean(),
        "priority_accuracy": valid_df["priority_correct"].mean(),
        "core_accuracy": valid_df["core_correct"].mean(),
        "fully_correct_rate": valid_df["fully_correct"].mean(),
        # Reasoning quality
        "avg_reasoning_coverage": valid_df["reasoning_coverage"].mean(),
        "good_reasoning_rate": valid_df["reasoning_quality_good"].mean(),
        # Calendar handling
        "calendar_mention_accuracy": valid_df["calendar_mention_correct"].mean(),
        # Draft response
        "draft_response_rate": valid_df["has_draft_response"].mean(),
        "avg_draft_length": valid_df["draft_response_length"].mean(),
    }

    return metrics


def calculate_decision_type_breakdown(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Break down performance by expected decision type."""

    valid_df = eval_df[~eval_df.get("error", pd.Series([None] * len(eval_df))).notna()].copy()

    if len(valid_df) == 0:
        return pd.DataFrame()

    breakdown = (
        valid_df.groupby("expected_decision")
        .agg(
            {
                "decision_correct": ["count", "sum", "mean"],
                "priority_correct": "mean",
                "core_correct": "mean",
                "reasoning_coverage": "mean",
            }
        )
        .round(3)
    )

    breakdown.columns = [
        "n",
        "n_correct",
        "decision_accuracy",
        "priority_accuracy",
        "core_accuracy",
        "avg_reasoning_coverage",
    ]

    return breakdown


def calculate_priority_breakdown(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Break down performance by expected priority level."""

    valid_df = eval_df[~eval_df.get("error", pd.Series([None] * len(eval_df))).notna()].copy()

    if len(valid_df) == 0:
        return pd.DataFrame()

    breakdown = (
        valid_df.groupby("expected_priority")
        .agg(
            {
                "priority_correct": ["count", "sum", "mean"],
                "decision_correct": "mean",
                "core_correct": "mean",
            }
        )
        .round(3)
    )

    breakdown.columns = [
        "n",
        "n_correct",
        "priority_accuracy",
        "decision_accuracy",
        "core_accuracy",
    ]

    return breakdown


def generate_triage_summary_report(eval_df: pd.DataFrame) -> str:
    """Generate a human-readable summary report for triage evaluation."""

    metrics = calculate_triage_metrics(eval_df)
    decision_breakdown = calculate_decision_type_breakdown(eval_df)
    priority_breakdown = calculate_priority_breakdown(eval_df)

    report = []
    report.append("=" * 80)
    report.append("INVITATION TRIAGE EVALUATION SUMMARY")
    report.append("=" * 80)
    report.append("")

    # Overall stats
    report.append(f"Total test cases: {len(eval_df)}")
    if metrics.get("n_errors", 0) > 0:
        report.append(f"Errors encountered: {metrics['n_errors']}")
    report.append(f"Valid evaluations: {metrics.get('n', 0)}")
    report.append("")

    if metrics.get("n", 0) == 0:
        report.append("No valid test cases to evaluate.")
        report.append("=" * 80)
        return "\n".join(report)

    # PRIMARY METRICS
    report.append("PRIMARY METRICS")
    report.append("-" * 80)
    report.append(
        f"Decision Accuracy:        {metrics['decision_accuracy']:.1%} "
        f"({int(metrics['decision_accuracy'] * metrics['n'])}/{metrics['n']})"
    )
    report.append(
        f"Priority Accuracy:        {metrics['priority_accuracy']:.1%} "
        f"({int(metrics['priority_accuracy'] * metrics['n'])}/{metrics['n']})"
    )
    report.append(
        f"Core Correct (Both):      {metrics['core_accuracy']:.1%} "
        f"({int(metrics['core_accuracy'] * metrics['n'])}/{metrics['n']})"
    )
    report.append(
        f"Fully Correct (+ Reason): {metrics['fully_correct_rate']:.1%} "
        f"({int(metrics['fully_correct_rate'] * metrics['n'])}/{metrics['n']})"
    )
    report.append("")

    # REASONING QUALITY
    report.append("REASONING QUALITY")
    report.append("-" * 80)
    report.append(f"Average Coverage of Expected Points: {metrics['avg_reasoning_coverage']:.1%}")
    report.append(f"Good Reasoning Rate (≥50% coverage): {metrics['good_reasoning_rate']:.1%}")
    report.append("")

    # CALENDAR HANDLING
    report.append("CALENDAR HANDLING")
    report.append("-" * 80)
    report.append(f"Calendar Mention Accuracy: {metrics['calendar_mention_accuracy']:.1%}")
    report.append("")

    # DRAFT RESPONSES
    report.append("DRAFT RESPONSES")
    report.append("-" * 80)
    report.append(f"Draft Response Rate:       {metrics['draft_response_rate']:.1%}")
    report.append(f"Average Draft Length:      {metrics['avg_draft_length']:.0f} characters")
    report.append("")

    # DECISION TYPE BREAKDOWN
    if len(decision_breakdown) > 0:
        report.append("PERFORMANCE BY DECISION TYPE")
        report.append("-" * 80)
        for decision_type in decision_breakdown.index:
            row = decision_breakdown.loc[decision_type]
            report.append(f"{decision_type.upper()}:")
            report.append(f"  Cases: {int(row['n'])}")
            report.append(f"  Decision Accuracy: {row['decision_accuracy']:.1%}")
            report.append(f"  Priority Accuracy: {row['priority_accuracy']:.1%}")
            report.append(f"  Core Accuracy:     {row['core_accuracy']:.1%}")
            report.append(f"  Avg Reasoning:     {row['avg_reasoning_coverage']:.1%}")
        report.append("")

    # PRIORITY LEVEL BREAKDOWN
    if len(priority_breakdown) > 0:
        report.append("PERFORMANCE BY PRIORITY LEVEL")
        report.append("-" * 80)
        for priority_level in priority_breakdown.index:
            row = priority_breakdown.loc[priority_level]
            report.append(f"{priority_level.upper()}:")
            report.append(f"  Cases: {int(row['n'])}")
            report.append(f"  Priority Accuracy: {row['priority_accuracy']:.1%}")
            report.append(f"  Decision Accuracy: {row['decision_accuracy']:.1%}")
            report.append(f"  Core Accuracy:     {row['core_accuracy']:.1%}")
        report.append("")

    report.append("=" * 80)

    return "\n".join(report)


def analyse_decision_errors(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Return DataFrame of all decision classification errors."""

    errors = eval_df[eval_df["decision_correct"] == False].copy()

    if len(errors) == 0:
        print("No decision errors found! 🎉")
        return pd.DataFrame()

    return errors[
        [
            "test_id",
            "description",
            "expected_decision",
            "actual_decision",
            "expected_priority",
            "actual_priority",
        ]
    ]


def analyse_priority_errors(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Return DataFrame of priority classification errors (where decision was correct)."""

    errors = eval_df[(eval_df["decision_correct"] == True) & (eval_df["priority_correct"] == False)].copy()

    if len(errors) == 0:
        print("No priority errors (with correct decision) found! 🎉")
        return pd.DataFrame()

    return errors[
        [
            "test_id",
            "description",
            "actual_decision",
            "expected_priority",
            "actual_priority",
        ]
    ]


def analyse_reasoning_gaps(eval_df: pd.DataFrame, coverage_threshold: float = 0.5) -> pd.DataFrame:
    """
    Return DataFrame of cases with poor reasoning coverage.

    Args:
        eval_df: Evaluation results DataFrame
        coverage_threshold: Minimum coverage to be considered "good" (default 0.5)
    """

    poor_reasoning = eval_df[(eval_df["reasoning_coverage"] < coverage_threshold)].copy()

    if len(poor_reasoning) == 0:
        print(f"No cases with reasoning coverage below {coverage_threshold:.0%}! 🎉")
        return pd.DataFrame()

    return poor_reasoning[
        [
            "test_id",
            "description",
            "actual_decision",
            "reasoning_coverage",
            "reasoning_missing_points",
        ]
    ]


def analyse_calendar_errors(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Return DataFrame of cases where calendar mention was incorrect."""

    errors = eval_df[eval_df["calendar_mention_correct"] == False].copy()

    if len(errors) == 0:
        print("No calendar mention errors found! 🎉")
        return pd.DataFrame()

    return errors[
        [
            "test_id",
            "description",
            "expected_calendar_mention",
            "actual_calendar_mention",
            "actual_decision",
        ]
    ]


def get_detailed_case_analysis(eval_df: pd.DataFrame, test_id: str) -> dict[str, Any]:
    """
    Get detailed analysis of a specific test case.

    Args:
        eval_df: Evaluation results DataFrame
        test_id: Test case ID to analyze

    Returns:
        Dictionary with detailed analysis
    """

    row = eval_df[eval_df["test_id"] == test_id]

    if len(row) == 0:
        return {"error": f"Test case {test_id} not found"}

    row = row.iloc[0]

    analysis = {
        "test_id": test_id,
        "description": row["description"],
        "expected": {
            "decision": row["expected_decision"],
            "priority": row["expected_priority"],
            "calendar_mention": row.get("expected_calendar_mention", None),
        },
        "actual": {
            "decision": row["actual_decision"],
            "priority": row["actual_priority"],
            "calendar_mention": row.get("actual_calendar_mention", None),
        },
        "correctness": {
            "decision": row["decision_correct"],
            "priority": row["priority_correct"],
            "core": row["core_correct"],
            "fully": row["fully_correct"],
        },
        "reasoning": {
            "coverage": row["reasoning_coverage"],
            "quality_good": row["reasoning_quality_good"],
            "missing_points": row.get("reasoning_missing_points", []),
            "actual_text": row.get("actual_reason", ""),
        },
        "draft_response": {
            "present": row.get("has_draft_response", False),
            "length": row.get("draft_response_length", 0),
            "text": row.get("actual_draft_response", ""),
        },
    }

    return analysis

# Evaluation Output

In [4]:
async def evaluate_and_report(
    test_cases: list[dict], persona: MinisterPersona, save_to_csv: str = None
) -> pd.DataFrame:
    """
    Run evaluation and print summary report.

    Args:
        test_cases: List of test case dictionaries
        persona: Minister persona to use
        save_to_csv: Optional path to save results CSV

    Returns:
        DataFrame with evaluation results
    """

    # Run evaluation
    eval_df = await run_triage_evaluation(test_cases, persona)

    # Print summary
    print("\n")
    print(generate_triage_summary_report(eval_df))

    # Save if requested
    if save_to_csv:
        eval_df.to_csv(save_to_csv, index=False)
        print(f"\nResults saved to {save_to_csv}")

    return eval_df

In [5]:
print("=" * 80)
print("TRIAGE EVALUATION")
print("=" * 80)
print()

# Convert test persona dict to MinisterPersona model
persona = MinisterPersona(**TEST_PERSONA)

# Run evaluation
eval_df = await evaluate_and_report(
    test_cases=TRIAGE_TEST_CASES,
    persona=persona,
)

TRIAGE EVALUATION

Running triage evaluation on 15 test cases...

[1/15] triage_001: AI Safety Summit - conflicting high priority justifies deferral
  ✓ CORE CORRECT: Decision=accept, Priority=high
  ⚠ Reasoning coverage: 25%

[2/15] triage_002: University research meeting conflicts with high-priority internal session
  ✓ PERFECT: Decision=defer, Priority=high

[3/15] triage_003: Quantum technology meeting fits perfectly in available Thursday slot
  ✓ PERFECT: Decision=accept, Priority=high

[4/15] triage_004: Evening reception violates morning preference - delegate
  ✗ INCORRECT:
    Decision: Expected 'delegate', got 'decline'
    Priority: Expected 'low', got 'medium'

[5/15] triage_005: Quantum breakthrough briefing - worth rescheduling comms meeting
  ✗ INCORRECT:
    Decision: Expected 'accept', got 'defer'

[6/15] triage_006: Nuclear innovation roundtable fits Friday morning perfectly
  ✓ CORE CORRECT: Decision=defer, Priority=high
  ⚠ Reasoning coverage: 25%

[7/15] triage_007:

CancelledError: 